# Fairness-Aware GPT-2 — QQP Training

Trains the paraphrase models from the report and produces the Table 2 numbers.
Scope is **QQP only** — that's where the fairness contribution is.

**First: Runtime → Change runtime type → T4 GPU.**

Run the cells **in order, top to bottom**. Each one depends on the last.

## 0. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 1. Set up

**Edit `REPO` to your repo URL before running.** Takes ~5 minutes.

Colab clones from GitHub, so anything you haven't pushed won't be here.

In [ ]:
REPO = "https://github.com/YOUR-USERNAME/fairness-aware-gpt2"   # <-- EDIT THIS

In [ ]:
!pip install -q uv

In [ ]:
!git clone -q $REPO fairness-repo

In [ ]:
%cd fairness-repo

In [ ]:
!uv sync --group train

In [ ]:
# Swap the CPU torch wheel (pinned for Streamlit Cloud) for the CUDA build.
!uv pip install -q torch --torch-backend=auto

In [ ]:
!uv run python -c "import torch; print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())" 

Must print `cuda: True`. If it says `False`, fix the runtime and re-run.

## 2. Download QQP

`--train-size 283011` matches the report's pair count (GLUE ships 363,846) and
cuts ~29% off every run. Dev stays at 40,430 — exactly the report's.

In [ ]:
!uv run scripts/download_data.py --only qqp --train-size 283011 --skip-test

## 3. Smoke test — do not skip

Two minutes. Catches setup problems before you spend two hours finding them.

In [ ]:
!uv run fairness-train --mode cda_reg --train data/quora-train.csv --dev data/quora-dev.csv --out /tmp/smoke --epochs 1 --train-subset 2000 --eval-subset 1000

Check the JSON line it prints:

- `dev_acc` around **0.6–0.7** is correct here — it's 1 epoch on 2,000 of 283,011 pairs.
  You're checking that it *runs*, not that it's good.
- `fairness_loss` must be **> 0**. If it's exactly `0.0`, no counterfactuals were
  generated and `cda_reg` has quietly degraded into plain CDA. Stop and check.

## 4. Train `cda_reg` — the model your app deploys

~2 hours. `--save-half` writes fp16 (~250MB) for Streamlit Cloud's memory limit.

**Keep this tab open and visible.** Free Colab kills idle sessions.

In [ ]:
!uv run fairness-train --mode cda_reg --train data/quora-train.csv --dev data/quora-dev.csv --out checkpoints/cda_reg --epochs 5 --lambda-fair 0.5 --save-half

## 5. Save results now

Writes `results/reproduced/cda_reg.json` — that's what makes your dashboard show
**your** numbers beside the paper's. Download it before starting anything else,
in case the session dies.

In [ ]:
!zip -qr reproduced.zip results/reproduced
from google.colab import files
files.download("reproduced.zip")

## 6. Push the model to Hugging Face

Needs a **Write** token from huggingface.co/settings/tokens.

In [ ]:
!uv run huggingface-cli login

In [ ]:
HF_USER = "YOUR-HF-USERNAME"   # <-- EDIT THIS

In [ ]:
!uv run scripts/push_to_hub.py --ckpt checkpoints/cda_reg --repo $HF_USER/fairness-gpt2-qqp

Then set this in your Streamlit app → Settings → Secrets:

```toml
MODEL_REPO = "YOUR-HF-USERNAME/fairness-gpt2-qqp"
```

Live predictions turn on.

---

## 7. Optional — the other two models

`cda_reg` alone is enough to deploy. These only add the full Table 2 comparison.
Run **one at a time**, re-downloading results after each.

In [ ]:
!uv run fairness-train --mode baseline --train data/quora-train.csv --dev data/quora-dev.csv --out checkpoints/baseline --epochs 5

In [ ]:
!uv run fairness-train --mode cda --train data/quora-train.csv --dev data/quora-dev.csv --out checkpoints/cda --epochs 5

In [ ]:
!zip -qr reproduced.zip results/reproduced
from google.colab import files
files.download("reproduced.zip")

## What to compare against

At **5 epochs**, your report's Table 4 gives a direct comparison:

| Mode | Dev acc | Subgroup gap | Flip rate |
|---|---|---|---|
| CDA | 0.8835 | 0.0433 | 0.0392 |
| CDA + Reg. | 0.8856 | 0.0512 | 0.0296 |

Table 4 has no 5-epoch baseline — compare that against the 10-epoch row
(0.8955 / 0.0365 / 0.0456) and expect lower accuracy.

**Your flip rate will not match.** The report gives substitution *counts*
(60/20/22) and three examples, not the lists — `identity.py` reconstructs them.
Different lexicon, different counterfactuals. Accuracy should land close; flip
rate is the number that moves.